# The Deutsch-Jozsa Algorithm

## The natural generalisation of Deutsch to $n$-bit input functions. Here the speedup over deterministic classical algorithms is **exponential**, making Deutsch–Jozsa the first proof that quantum computation can outperform classical by an exponential factor (in the query model).

# 1. The problem

## Given an oracle for $f : \{0,1\}^n \to \{0,1\}$ promised to be either

## - **constant** (the same value for every input), or
## - **balanced** (equals 0 on exactly half the inputs and 1 on the other half),

## decide which.

## Classically, the worst case requires $2^{n-1} + 1$ queries ($2^{n-1}$ matching values are not enough to rule out a balanced $f$). Quantumly: **one query**.

# 2. The algorithm

## 1. Prepare $n$ input qubits in $|0\rangle^{\otimes n}$ and one ancilla in $|1\rangle$.
## 2. Apply $H$ to **all** $n+1$ qubits. State: $|+\rangle^{\otimes n}|-\rangle$.
## 3. Apply $U_f$.
## 4. Apply $H^{\otimes n}$ to the input register.
## 5. Measure the input register.

## If the outcome is $\mathbf{0} = 00\dots 0$, the function is **constant**. Any other outcome means **balanced**.

# 3. Why it works

## After step 2 the input register is

$$ \Large  \frac{1}{\sqrt{2^n}} \sum_{x \in \{0,1\}^n} |x\rangle. $$

## After phase kickback (step 3) it becomes

$$ \Large  \frac{1}{\sqrt{2^n}} \sum_x (-1)^{f(x)} |x\rangle. $$

## Apply $H^{\otimes n}$ and use the identity $H^{\otimes n}|x\rangle = \tfrac{1}{\sqrt{2^n}}\sum_y (-1)^{x\cdot y}|y\rangle$:

$$ \Large  \frac{1}{2^n} \sum_y \Big(\sum_x (-1)^{f(x) + x \cdot y}\Big) |y\rangle. $$

## The amplitude of $|\mathbf{0}\rangle$ is $\tfrac{1}{2^n}\sum_x (-1)^{f(x)}$. If $f$ is constant, this sum is $\pm 1$, so the outcome is $\mathbf{0}$ with probability 1. If $f$ is balanced, the signs cancel exactly and the amplitude is 0 — outcome $\mathbf{0}$ is impossible.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

def dj_oracle(case, n):
    qc = QuantumCircuit(n + 1)
    if case == 'constant':
        pass                              # f(x) = 0 always
    elif case == 'balanced':
        for q in range(n):
            qc.cx(q, n)                   # f(x) = parity(x)
    return qc

def deutsch_jozsa(case, n=4, shots=1024):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.compose(dj_oracle(case, n), inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    sim = AerSimulator()
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return counts

print('constant :', deutsch_jozsa('constant'))
print('balanced :', deutsch_jozsa('balanced'))

constant : {'0000': 1024}
balanced : {'1111': 1024}


# 4. The speedup, more carefully

## The exponential separation holds against **deterministic** classical algorithms. Against **randomised** classical algorithms, Deutsch–Jozsa is solvable in $O(1)$ queries with bounded error: sample $f$ at $k$ random inputs and report "balanced" the moment two outputs disagree. So strictly, the BQP-vs-P separation needs more (it comes with Bernstein–Vazirani, Simon, and ultimately Shor).

## Still, Deutsch–Jozsa is the clearest pedagogical example of how Hadamard + oracle + Hadamard, with the right phase kickback, makes interference do the work of exponentially many classical evaluations.

# Recap

## - Promise problem: $f$ is constant or balanced.
## - Quantum: 1 query. Classical (deterministic): $\Omega(2^n)$ queries.
## - Mechanism: phase kickback into a uniform superposition, then $H^{\otimes n}$ to interfere.
## - First exponential quantum speedup (in the deterministic-query model).